# 04 — Embeddings (AraVec) — Kordy

**Goal:** turn the 19,999 words of our vocabulary (`data/word_index_top20k.json`) into one **20,000 × 100** embedding matrix

## 1. Setup

In [1]:
import sys, subprocess, json, re, zipfile, urllib.request
from pathlib import Path

# Install gensim only if it is missing (it is not part of the default Colab / Anaconda setup)
try:
    import gensim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])
    import gensim
from gensim.models import Word2Vec

import numpy as np
import pandas as pd
from IPython.display import display

# Find the repo root: the folder that contains data/ (works from notebooks/ or from the root)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
assert (ROOT / "data" / "word_index_top20k.json").exists(), f"Run this notebook inside the repo (looked in {ROOT})"

DATA_DIR = ROOT / "data"
RES_DIR = ROOT / "results"
EMB_DIR = ROOT / "embeddings"            # blocked by .gitignore -> never pushed
EMB_DIR.mkdir(exist_ok=True)

MODEL_NAME = "full_uni_cbow_100_twitter"  # Twitter, CBOW, 100 dims, unigrams
MODEL_URL = f"https://bakrianoo.ewr1.vultrobjects.com/aravec/{MODEL_NAME}.zip"   # link from the AraVec README
EMBED_DIM = 100

print("repo root:", ROOT)
print("gensim   :", gensim.__version__, "| numpy:", np.__version__, "| pandas:", pd.__version__)

repo root: c:\Users\ahmad\OneDrive\Desktop\arabic-tweet-emotion-bilstm-main\arabic-tweet-emotion-bilstm-main
gensim   : 4.4.0 | numpy: 2.4.4 | pandas: 2.3.3


## 2. Download and load AraVec

In [2]:
import time, urllib.error

def download(url, dest, tries=30):
    # Download in pieces and resume after a dropped connection (the zip is ~960 MB)
    part = dest.with_name(dest.name + ".part")           # unfinished download
    for attempt in range(1, tries + 1):
        have = part.stat().st_size if part.exists() else 0
        headers = {"Range": f"bytes={have}-"} if have else {}   # ask only for the missing part
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
                if have and r.status != 206:              # server ignored Range -> start again
                    have = 0
                total = have + int(r.headers["Content-Length"])
                with open(part, "ab" if have else "wb") as f:
                    while chunk := r.read(1 << 20):       # 1 MB at a time
                        f.write(chunk)
                        have += len(chunk)
                        print(f"\r{have / 1e6:,.0f} / {total / 1e6:,.0f} MB", end="")
            if have >= total:
                part.replace(dest)
                print("\nDownload complete.")
                return
        except urllib.error.HTTPError as e:
            if e.code == 416:                             # nothing left to download: already complete
                part.replace(dest)
                return
            print(f"\nAttempt {attempt}: {e} -> retrying")
        except Exception as e:
            print(f"\nAttempt {attempt} stopped at {have / 1e6:,.0f} MB: {e} -> resuming")
        time.sleep(5)
    raise RuntimeError("Download keeps failing: download the zip in your browser and put it in embeddings/")

model_file = EMB_DIR / f"{MODEL_NAME}.mdl"
zip_path = EMB_DIR / f"{MODEL_NAME}.zip"

if not model_file.exists():
    if zip_path.exists() and not zipfile.is_zipfile(zip_path):
        print("Removing a broken zip from an earlier try")
        zip_path.unlink()
    if not zip_path.exists():
        print("Downloading", MODEL_URL)
        download(MODEL_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        print("Files in the zip:", z.namelist())
        z.extractall(EMB_DIR)          # .mdl + its .npy files must stay in the same folder

# AraVec is a full gensim Word2Vec model; we only need its word vectors (.wv)
aravec = Word2Vec.load(str(model_file)).wv

print("AraVec vocabulary:", f"{len(aravec.key_to_index):,}", "words")
print("vector size      :", aravec.vector_size)
assert aravec.vector_size == EMBED_DIM

C:\Users\ahmad\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gensim\utils.py:1460: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return _pickle.load(f, encoding='latin1')  # needed because loading from S3 doesn't support readline()


AraVec vocabulary: 1,259,756 words
vector size      : 100


## 3. Load our vocabulary

In [3]:
with open(DATA_DIR / "word_index_top20k.json", encoding="utf-8") as f:
    word_index = json.load(f)              # {word: index}, built by the Keras tokenizer on train only

with open(DATA_DIR / "preprocessing_config.json", encoding="utf-8") as f:
    config = json.load(f)

NUM_ROWS = config["embed_input_dim"]       # 20000 = 19,999 words + row 0 for padding

# Check that what we received is what the task sheet says
indices = sorted(word_index.values())
assert NUM_ROWS == 20000, NUM_ROWS
assert len(word_index) == NUM_ROWS - 1 == 19999, len(word_index)
assert indices == list(range(1, NUM_ROWS)), "indices must be exactly 1 .. 19,999"

print("words:", len(word_index), "| indices:", indices[0], "to", indices[-1], "| matrix rows:", NUM_ROWS)
print("first 10:", list(word_index.items())[:10])

words: 19999 | indices: 1 to 19999 | matrix rows: 20000
first 10: [('الاوليمبياد', 1), ('علي', 2), ('خايفه', 3), ('اللي', 4), ('الله', 5), ('لا', 6), ('ما', 7), ('مش', 8), ('مصر', 9), ('الحب', 10)]


## 4. Lookup rules

Our words are already cleaned with Salma's `clean_text`. AraVec's own normalization (`clean_str` in the AraVec repo) differs in a few places, so a word can be in AraVec under a slightly different spelling.

**Try 1:** look the word up as it is.
**Try 2:** if it is missing, rewrite it the way AraVec normalized its training tweets and look it up again.

AraVec rules that can change one of our words (checked in `clean_str`):

| AraVec rule | Already done by our `clean_text`? |
|---|---|
| أ إ آ → ا | yes |
| remove tashkeel | yes |
| ة → ه | **no** |
| ى → ي | **no** |
| a letter repeated 2+ times → kept twice; then وو → و, يي → ي, اا → ا | partly (we turn 3+ repeats into 1, but leave doubles) |

The other `clean_str` rules remove punctuation or fix spaces, and our words have neither, so they are skipped. We only change the **lookup key** — the word stays the same in our vocabulary and in Salma's sequences.

In [4]:
DOUBLE_RE = re.compile(r"(.)\1+")              # any letter repeated 2 or more times

def aravec_form(word):
    # Rewrite one of our words the way AraVec normalized its corpus (rules from AraVec's clean_str)
    w = DOUBLE_RE.sub(r"\1\1", word)           # long repeats -> two letters
    w = w.replace("وو", "و").replace("يي", "ي").replace("اا", "ا")   # these doubles -> one letter
    w = w.replace("ة", "ه")                    # taa marbuta -> haa
    w = w.replace("ى", "ي")                    # alef maqsura -> yaa
    return w

for w in ["خايفه", "حياة", "على", "مستنيي", "موجوود"]:
    print(f"{w:>10} -> {aravec_form(w)}")

     خايفه -> خايفه
      حياة -> حياه
       على -> علي
    مستنيي -> مستني
    موجوود -> موجود


## 5. Build the 20,000 × 100 matrix

In [5]:
embedding_matrix = np.zeros((NUM_ROWS, EMBED_DIM), dtype=np.float32)   # row 0 (padding) and missing words stay zeros

lookup_log = []                                  # one row per word: how it was found
for word, i in word_index.items():
    if word in aravec.key_to_index:              # try 1: as it is
        embedding_matrix[i] = aravec[word]
        lookup_log.append((i, word, word, "as_is"))
        continue
    key = aravec_form(word)
    if key != word and key in aravec.key_to_index:   # try 2: AraVec's normalization
        embedding_matrix[i] = aravec[key]
        lookup_log.append((i, word, key, "aravec_norm"))
    else:
        lookup_log.append((i, word, key, "missing"))  # zero vector, as in the paper (Sec. 3.6)

lookup_df = pd.DataFrame(lookup_log, columns=["index", "word", "aravec_key", "found_by"]).sort_values("index")
print(lookup_df["found_by"].value_counts())

found_by
as_is          18721
missing         1262
aravec_norm       16
Name: count, dtype: int64


### Checks

In [6]:
assert embedding_matrix.shape == (20000, 100)
assert embedding_matrix.dtype == np.float32
assert not embedding_matrix[0].any(), "row 0 must stay zeros (padding)"
assert not np.isnan(embedding_matrix).any()

# Row i must be the vector of the word whose index is i
for _, r in lookup_df[lookup_df["found_by"] != "missing"].sample(20, random_state=42).iterrows():
    assert np.allclose(embedding_matrix[r["index"]], aravec[r["aravec_key"]])

# A missing word has a zero row, a found word does not
found = lookup_df.set_index("index")["found_by"] != "missing"
nonzero = embedding_matrix.any(axis=1)
assert (nonzero[1:] == found.sort_index().to_numpy()).all()

print("All checks passed.")

All checks passed.


## 6. Save the matrix (`.npz`)

In [7]:
out_path = DATA_DIR / "embedding_matrix.npz"
np.savez_compressed(out_path, embedding_matrix=embedding_matrix)

# Reload it exactly the way Yassin and Mahmoud will
reloaded = np.load(out_path)["embedding_matrix"]
assert reloaded.shape == (20000, 100) and np.array_equal(reloaded, embedding_matrix)
print("Saved", out_path.relative_to(ROOT), "|", f"{out_path.stat().st_size / 1e6:.1f} MB")

Saved data\embedding_matrix.npz | 7.0 MB


## 7. Coverage

In [8]:
n_words = len(word_index)
n_as_is = int((lookup_df["found_by"] == "as_is").sum())
n_norm = int((lookup_df["found_by"] == "aravec_norm").sum())
n_found = n_as_is + n_norm

# Occurrence level: every word occurrence in the train tweets (CLEAN column)
train_df = pd.read_csv(DATA_DIR / "train_clean.csv", encoding="utf-8")
tokens = [w for t in train_df["CLEAN"].fillna("") for w in t.split()]
has_vec = {w for w, i in word_index.items() if nonzero[i]}      # words whose row is not zeros

n_tokens = len(tokens)
n_tok_vocab = sum(w in word_index for w in tokens)               # inside the top 20k
n_tok_vec = sum(w in has_vec for w in tokens)                    # get an AraVec vector

coverage = pd.DataFrame([
    ("vocabulary words", n_words),
    ("found as is (try 1)", n_as_is),
    ("found after AraVec normalization (try 2)", n_norm),
    ("found in total", n_found),
    ("missing (zero vector)", n_words - n_found),
    ("word coverage", f"{n_found / n_words:.2%}"),
    ("train word occurrences", n_tokens),
    ("occurrences inside the top-20k vocabulary", n_tok_vocab),
    ("occurrences that get an AraVec vector", n_tok_vec),
    ("occurrence coverage (of all train occurrences)", f"{n_tok_vec / n_tokens:.2%}"),
    ("occurrence coverage (of in-vocabulary occurrences)", f"{n_tok_vec / n_tok_vocab:.2%}"),
], columns=["statistic", "value"]).astype({"value": str})   # str: keep counts as whole numbers

RES_DIR.mkdir(exist_ok=True)
coverage.to_csv(RES_DIR / "embedding_coverage.csv", index=False, encoding="utf-8")
display(coverage)

word_cov, occ_cov = n_found / n_words, n_tok_vec / n_tokens
print(f"\nWord coverage      : {word_cov:.2%}")
print(f"Occurrence coverage: {occ_cov:.2%}")
if min(word_cov, occ_cov) < 0.50:
    print("WARNING: coverage is below 50% -> tell the team now")

,statistic,value
0,vocabulary words,19999
1,found as is (try 1),18721
2,found after AraVec normalization (try 2),16
3,found in total,18737
4,missing (zero vector),1262
5,word coverage,93.69%
6,train word occurrences,50374
7,occurrences inside the top-20k vocabulary,49870
8,occurrences that get an AraVec vector,48246
9,occurrence coverage (of all train occurrences),95.78%



Word coverage      : 93.69%
Occurrence coverage: 95.78%


### Words worth checking for the discussion

In [9]:
# Kept negation words and laughter (هه): preprocessing kept them on purpose, so do they get a vector?
check = config["kept_negation_words"] + ["هه"]
status = lookup_df.set_index("word").reindex(check)[["index", "found_by"]]
status["found_by"] = status["found_by"].fillna("not in our vocabulary")   # never seen in train
display(status)

,index,found_by
word,,
لا,6.0,as_is
لم,33.0,as_is
لن,85.0,as_is
ما,7.0,as_is
ليس,64.0,as_is
ولا,11.0,as_is
فلا,537.0,as_is
وما,161.0,as_is
ليست,768.0,as_is


In [10]:
# The most frequent missing words in train: what kind of words AraVec does not know
counts = pd.Series(tokens).value_counts()
missing = lookup_df[lookup_df["found_by"] == "missing"].copy()
missing["train_count"] = missing["word"].map(counts).fillna(0).astype(int)
missing = missing.sort_values("train_count", ascending=False)
missing[["word", "index", "train_count"]].to_csv(RES_DIR / "embedding_missing_words.csv", index=False, encoding="utf-8")
missing[["word", "index", "train_count"]].head(30)

,word,index,train_count
37,d,38,69
182,rio,183,24
438,h,439,12
524,a,525,11
576,salyy,577,10
708,lt,709,9
775,doaaelsebaii,776,8
796,staracarabia,797,8
870,m,871,7
915,k,916,7


## 8. Sanity check: nearest words

In [11]:
PROBES = {"فرحان": "happy", "زعلان": "upset", "خايف": "afraid", "بحبك": "I love you", "غضبان": "angry", "حزين": "sad"}
TOP_N = 10

rows = []
for word, meaning in PROBES.items():
    key = word if word in aravec.key_to_index else aravec_form(word)
    if key not in aravec.key_to_index:
        print("not in AraVec:", word)
        continue
    for rank, (neigh, score) in enumerate(aravec.most_similar(key, topn=TOP_N), start=1):
        rows.append({"word": word, "meaning": meaning, "rank": rank, "neighbour": neigh, "cosine": round(float(score), 3)})

nearest = pd.DataFrame(rows)
nearest.to_csv(RES_DIR / "nearest_words.csv", index=False, encoding="utf-8")

# One line per probe word, easy to paste on a slide
nearest.groupby(["word", "meaning"], sort=False)["neighbour"].apply(lambda s: "، ".join(s)).to_frame("nearest words")

,,nearest words
word,meaning,
فرحان,happy,وفرحان، مبسوط، سعيد، فرحاان، شمتان، مبسووط، مت...
زعلان,upset,زعﻻن، زعلاان، متضايق، مضايق، متنرفز، زعلانه، م...
خايف,afraid,خايفه، وخايف، خاايف، متعود، زعلان، بخاف، ناوي،...
بحبك,I love you,بحببك، بعشقك، بحبكك، بحبگ، بحبه، وبحبك، باحبك،...
غضبان,angry,غضبا*صحيح، غضبانامين، مئسوف، الجالس»، اذان*، ب...
حزين,sad,حزيين، حزينه، حززين، وحيد، موجوع، مجروح، مكسور...


## 9. Numbers to send

In [12]:
print(f"Matrix: data/embedding_matrix.npz, key 'embedding_matrix', shape {embedding_matrix.shape}, float32")
print(f"AraVec model: {MODEL_NAME} ({len(aravec.key_to_index):,} words, {EMBED_DIM} dims)")
print(f"Found as is: {n_as_is:,} | after AraVec normalization: {n_norm:,} | missing: {n_words - n_found:,} of {n_words:,}")
print(f"Word coverage: {word_cov:.2%} | occurrence coverage (train): {occ_cov:.2%}")

Matrix: data/embedding_matrix.npz, key 'embedding_matrix', shape (20000, 100), float32
AraVec model: full_uni_cbow_100_twitter (1,259,756 words, 100 dims)
Found as is: 18,721 | after AraVec normalization: 16 | missing: 1,262 of 19,999
Word coverage: 93.69% | occurrence coverage (train): 95.78%
